Init code block

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

Step 2: simple vs log on real data 

In [ ]:
from quantlib.data import load_prices
from quantlib.metrics import simple_returns, log_returns

close = load_prices("SPY", "2010-01-01", "2024-12-31")["Close"]   # reads the cache
r = simple_returns(close)
l = log_returns(close)

print("days:", len(r))
print("mean daily  simple:", round(r.mean(), 6), " log:", round(l.mean(), 6))
print("max |simple - log|:", round((r - l).abs().max(), 5), "on", (r - l).abs().idxmax().date())
print("2020-03-16  simple:", round(r.loc['2020-03-16'], 4), " log:", round(l.loc['2020-03-16'], 4))

days: 3772
mean daily  simple: 0.000568  log: 0.00051
max |simple - log|: 0.00646 on 2020-03-16
2020-03-16  simple: -0.1094  log: -0.1159


Step 3: additivity and compounding 

In [3]:
import numpy as np
from quantlib.metrics import cumulative_return, equity_curve

# (a) Log returns ADD across time -> they sum to the total log price relative.
print("sum of log returns :", round(l.sum(), 4))
print("ln(last / first)   :", round(np.log(close.iloc[-1] / close.iloc[0]), 4))   # equal

# (b) Simple returns COMPOUND -> equity curve endpoint == 1 + cumulative_return.
eq = equity_curve(r)                       # growth of $1
print("equity end ($1->)  :", round(eq.iloc[-1], 4))
print("1 + cumulative     :", round(1 + cumulative_return(r), 4))                 # equal
print("raw price ratio    :", round(close.iloc[-1] / close.iloc[0], 4))           # also equal

sum of log returns : 1.9224
ln(last / first)   : 1.9224
equity end ($1->)  : 6.8377
1 + cumulative     : 6.8377
raw price ratio    : 6.8377


Step 4: the honest annualized number

In [4]:
from quantlib.metrics import annualized_return, TRADING_DAYS

cagr  = annualized_return(r)
arith = r.mean() * TRADING_DAYS
print("geometric CAGR  :", f"{cagr*100:.2f}%")     # ~13.70%
print("arithmetic x252 :", f"{arith*100:.2f}%")    # ~14.31%  (too high)
print("drag            :", f"{(arith - cagr)*100:.2f}%")

# the precise bridge: per-day, mean(simple) - mean(log) ~ sigma^2 / 2
print("mean(simple)-mean(log):", round(r.mean() - l.mean(), 7))
print("sigma^2 / 2 (daily)   :", round(r.var() / 2, 7))     # nearly identical


geometric CAGR  : 13.70%
arithmetic x252 : 14.31%
drag            : 0.60%
mean(simple)-mean(log): 5.8e-05
sigma^2 / 2 (daily)   : 5.77e-05
